# 02 — Text Preprocessing

Goal: Clean raw tweet text for TF-IDF vectorization.

**Pipeline:** Lowercase → Remove URLs → Remove @mentions → Keep hashtag words → Remove punctuation → Remove stopwords → Filter short tokens

In [1]:
import pandas as pd
import re

df = pd.read_csv('../data/Tweets.csv')
df['tweet_created'] = pd.to_datetime(df['tweet_created'], utc=True)
print("Loaded:", df.shape)

Loaded: (14640, 15)


## 2.1 Stopword List

Custom stopword list tuned for tweet text — includes common Twitter tokens like 'rt', 'via', 'co'.

In [2]:
STOPWORDS = set([
    'i','me','my','myself','we','our','you','your','he','him','his',
    'she','her','it','its','they','them','their','am','is','are','was',
    'were','be','been','have','has','had','do','does','did','will','would',
    'could','should','may','might','a','an','the','and','but','or','for',
    'so','at','by','from','in','of','on','to','up','with','as','just',
    'also','not','no','very','more','all','this','that','what','which',
    'there','here','when','where','how','get','got','can','its',
    # Twitter-specific
    'rt','co','via','amp','http','https','www'
])
print("Stopwords defined:", len(STOPWORDS), "tokens")

Stopwords defined: 81 tokens


## 2.2 Preprocessing Function

In [3]:
def preprocess_tweet(text):
    """
    Clean a single tweet:
    1. Lowercase
    2. Remove URLs
    3. Remove @mentions (keep hashtag words)
    4. Remove punctuation/numbers
    5. Remove stopwords + short tokens
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)                 # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)           # '#delay' → 'delay'
    text = re.sub(r'[^a-z\s]', ' ', text)           # remove non-alpha
    tokens = [t for t in text.split()
              if t not in STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_tweet)

print("=== PREPROCESSING EXAMPLES ===\n")
samples = df.groupby('airline_sentiment').first()[['text','clean_text']]
for sent, row in samples.iterrows():
    print(f"Sentiment : {sent}")
    print(f"Original  : {row['text'][:100]}")
    print(f"Cleaned   : {row['clean_text'][:100]}")
    print()

=== PREPROCESSING EXAMPLES ===

Sentiment : negative
Original  : @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp;
Cleaned   : really aggressive blast obnoxious entertainment guests faces little recourse

Sentiment : neutral
Original  : @VirginAmerica What @dhepburn said.
Cleaned   : said

Sentiment : positive
Original  : @VirginAmerica plus you've added commercials to the experience... tacky.
Cleaned   : plus added commercials experience tacky



## 2.3 Encode Target Labels

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['airline_sentiment'])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# negative=0, neutral=1, positive=2

# Check empty texts after cleaning
empty = (df['clean_text'].str.strip() == '').sum()
print(f"\nEmpty texts after cleaning: {empty}")
if empty > 0:
    df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
    print(f"Removed {empty} empty rows. Final shape: {df.shape}")

df[['clean_text','label','airline_sentiment','tweet_created',
    'airline','retweet_count']].to_csv('../data/preprocessed.csv', index=False)
print("\nSaved: ../data/preprocessed.csv")

Label mapping: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}

Empty texts after cleaning: 30
Removed 30 empty rows. Final shape: (14610, 17)

Saved: ../data/preprocessed.csv
